# Run a simple ASPECT case

This notebook shows how to run a case with gdmate and find its results. To run the case, you also need a working ASPECT executable.

## 1. Pick your case folder

Put the parameter file and the input files it uses in the same folder. For example:

```text
my_case/
  case.prm
  case.wb
```

Note that a case would also needs a `.wb` file if its parameters use one.

The settings below point to an example with wb file using WorldBuilder version 1.1 and the Release build of ASPECT version 3.2.
You can use your own case folder and parameter file instead.

In [ ]:
from pathlib import Path
from uuid import uuid4

from gdmate.aspect_tools import run_aspect_case

case_directory = Path("../data/mow_aspect_example_wb11")
parameter_file = "case.prm"
aspect_executable = Path(
    "/home/lochy/Softwares/aspect/build_reaction_kinetics_development/aspect-release"
)

## 2. Run ASPECT

Check the path in `aspect_executable`, then run the cell below. The run can take time, depending on the case.

ASPECT reads inputs from the case folder. Each run writes to a new folder named `output-...`, so earlier results are kept. ASPECT saves a copy of the parameters it used in the output folder. Your original parameter file stays as it was.

In [ ]:
result = run_aspect_case(
    aspect_executable,
    parameter_file,
    case_directory=case_directory,
    output_directory=f"output-{uuid4().hex}",
)
print("Exit code:", result.returncode)
print("Output folder:", result.output_directory)
print("Standard output log:", result.stdout_path)
print("Error log:", result.stderr_path)

## 3. Find the results

An exit code of `0` means ASPECT finished successfully. For any other code, open the two log files to see what happened.

If your case writes visualization files in VTU format, look for `.pvtu` files in the `solution` folder.

In [ ]:
if result.returncode != 0:
    print("ASPECT stopped with an error. Read the logs shown above.")
else:
    solution_directory = result.output_directory / "solution"
    snapshots = sorted(solution_directory.glob("*.pvtu"))

    if snapshots:
        for snapshot in snapshots:
            print(snapshot)
    else:
        print(f"No PVTU snapshots found in {solution_directory}.")

## 4. Load pressure and temperature

Open the last snapshot from the run above. It holds the mesh and the values at its points. 

ASPECT stores pressure as `p` in pascals and temperature as `T` in kelvin. 

Note that a flat 2-D mesh still has a third coordinate with zero values, so we take just `x` and `y` (first two columns) for the next steps.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from gdmate.aspect_tools import PointFieldSampler2D, read_point_fields
from gdmate.visualization.pyvista_vis import pv_plot_2d

if result.returncode != 0:
    raise RuntimeError(f"ASPECT failed. See {result.stderr_path}")
if not snapshots:
    raise FileNotFoundError("This run did not write a PVTU snapshot.")

mesh, fields = read_point_fields(snapshots[-1], ("p", "T"))
sampler = PointFieldSampler2D(mesh.points[:, :2], fields)
x_min, x_max, y_min, y_max = mesh.bounds[:4]

print(f"{mesh.n_points:,} points in {snapshots[-1].name}")

fig, ax = plt.subplots()
pv_plot_2d(
    mesh,
    "T",
    bounds=[x_min, x_max, y_min, y_max],
    ax=ax,
    colorbar=True,
    scalar_bar_args={"title": "Temperature (K)"},
)
ax.set(xlabel="X (m)", ylabel="Y (m)", title="Temperature on the ASPECT mesh")
plt.show()

## 5. Sample fields at two points

Pick two points in the box. The sampler uses linear interpolation by default to estimate pressure and temperature at each point. It builds triangles between the mesh points on the first call and saves them for later calls. Use `method="nearest"` if you want the value from the closest mesh point.

In [ ]:
x_middle = (x_min + x_max) / 2
box_height = y_max - y_min
query_points = [
    [x_middle, y_min + 0.25 * box_height],
    [x_middle, y_min + 0.75 * box_height],
]
samples = sampler.sample(query_points)

for (x, y), pressure, temperature in zip(
    query_points, samples["p"], samples["T"], strict=True
):
    print(f"({x / 1e3:.0f}, {y / 1e3:.0f}) km: {pressure / 1e9:.2f} GPa, {temperature:.1f} K")

## 6. Put temperature on a regular grid

Choose evenly spaced `x` and `y` coordinates. For nearest neighbor, the sampler reuses its search tree to find the closest mesh point to each grid point. For linear interpolation, it reuses the triangles from the previous step. The linear method uses the three corners of each triangle to estimate temperatures inside it. Points outside the triangles are left blank.

Both results have one row per `y` value and one column per `x` value.

In [ ]:
x_grid = np.linspace(x_min, x_max, 101)
y_grid = np.linspace(y_min, y_max, 101)
temperature_nearest = sampler.grid(x_grid, y_grid, method="nearest")["T"]
temperature_sampler_linear = sampler.grid(x_grid, y_grid, method="linear")["T"]

print("Grid shape:", temperature_nearest.shape)

## 7. Compare the three grids

Use the same mesh points and grid to call SciPy's `griddata` directly. Its linear result should match the sampler's linear result, while the sampler 's nearest-neighbor result is a little bit coarser. 

Here, we plot all three grids with the same color scale so their temperature patterns are easy to compare.

Let $N$ be the number of source points and $M$ the number of points on the output grid. These are approximate costs for 2-D data.

| Approach | Build the interpolator | Interpolate on the grid |
| --- | --- | --- |
| Sampler nearest neighbor | Build a KD-tree once, $O(N \log N)$. | $O(M \log N)$|
| Sampler linear | Build a Delaunay triangulation once, roughly $O(N \log N)$ for well-behaved points. | Depends on the mesh and query locations|

[Qhull performance notes](https://qhull.org/html/qh-code.htm) · [SciPy linear interpolator](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.LinearNDInterpolator.html)

In [ ]:
from scipy.interpolate import griddata

grid_x, grid_y = np.meshgrid(x_grid, y_grid)
temperature_griddata = griddata(
    sampler.points, sampler.fields["T"], (grid_x, grid_y), method="linear"
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
temperature_min = float(sampler.fields["T"].min())
temperature_max = float(sampler.fields["T"].max())
for ax, temperature, title in zip(
    axes,
    (temperature_nearest, temperature_sampler_linear, temperature_griddata),
    ("Sampler nearest neighbor", "Sampler linear", "SciPy griddata linear"),
    strict=True,
):
    image = ax.pcolormesh(
        x_grid / 1e3, y_grid / 1e3, temperature,
        shading="auto", vmin=temperature_min, vmax=temperature_max,
    )
    ax.set(xlabel="X (km)", ylabel="Y (km)", title=title, aspect="equal")

fig.colorbar(image, ax=axes, label="Temperature (K)")
plt.show()